## Lab 2 - ML
### Imports

In [20]:
import torch
import torchvision
from torchvision.io import decode_image
import torchcam
import json

print(torch.__version__)
print(torchvision.__version__)
print(torchcam.__version__)
print("CUDA available:", torch.cuda.is_available())

2.6.0+cu124
0.21.0+cu124
0.3.1
CUDA available: True


In [21]:
from torchvision.models import get_model, get_model_weights

weights = get_model_weights("resnet50").DEFAULT
model = get_model("resnet50", weights=weights).eval()

print("Model loaded")
print("Input preprocessing:", weights.transforms())

Model loaded
Input preprocessing: ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [22]:
with open("imagenet_class_index.json", "r") as f:
    class_index = json.load(f)

# Quick sanity check
print(class_index["6"])
print(class_index["321"])
print(class_index["973"])
print("Total classes:", len(class_index))

['n01498041', 'stingray']
['n02276258', 'admiral']
['n09256479', 'coral_reef']
Total classes: 1000


### Image decoder

In [23]:
img = decode_image("admiralbutterfly.jpg")
print(img.shape)
print(img.dtype)

torch.Size([3, 640, 640])
torch.uint8


### Preprocessing

In [24]:
preprocess = weights.transforms()

input_tensor = preprocess(img)
print(input_tensor.shape)
print(input_tensor.dtype)

torch.Size([3, 224, 224])
torch.float32


In [25]:
batch = input_tensor.unsqueeze(0)
print(batch.shape)

torch.Size([1, 3, 224, 224])


In [26]:
model = model.to("cuda")

with torch.no_grad():
    output = model(batch.to("cuda"))

print(output.shape)

torch.Size([1, 1000])


In [27]:
probs = torch.softmax(output.squeeze(0), dim=0)
print(probs.shape)
print(probs.sum())

torch.Size([1000])
tensor(1.0000, device='cuda:0')


### Prediction

In [28]:
def predict(probs): 
    index = probs.argmax().item()
    name = class_index[str(index)][1]
    confidence = probs[index].item()
    return name, confidence

name, confidence = predict(probs)
print(f"The image is predicted as {name} with a confidence of {confidence:.2%}")

The image is predicted as admiral with a confidence of 46.76%


In [ ]:
top5 = torch.topk(probs, 5)
for i in range(5):
    index = top5.indices[i].item()
    name = class_index[str(index)][1]
    confidence = top5.values[i].item()
    print(f"{name}: {confidence:.2%}")

admiral: 46.76%
ringlet: 1.20%
monarch: 0.29%
lycaenid: 0.17%
grasshopper: 0.11%
